## The saturation spread and Table I

The paper reports the across-layer correlation between robustness αℓ and sensitivity βℓ for three architectures on two degradation axes, but it does not present a negative sign as evidence: CKA saturates at the clean end of every sweep, and for perfectly linear curves the slope is forced to be β = (c_high − α)/s_high, so r(α, β) follows from an identity in two numbers, the correlation r_c = r(α, c_high) and the **saturation spread** f = sd(c_high)/sd(α) (Sec. "Saturation Spread"). This chapter rebuilds every row of Table I from the shipped mean curves and fit tables with `se_probe.profiles.tradeoff_summary`, draws the tradeoff scatter of Fig. 7, and shows what the identity permits.

Runtime: under 30 seconds on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _rel(p):
    """Repo-relative path for display (absolute only if outside the repo)."""
    try:
        return p.relative_to(REPO)
    except ValueError:
        return p

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, MODEL_LABELS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import numpy as np
import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {_rel(DATA_DIR)}")
print(f"Paper tables from: {_rel(TABLES)}")

## Noise-axis rows

The MUSE row is recomputed from the raw 18-noise mean-CKA grid (`cka_heatmap_values_raw_jobB18.csv`, 824 × 18 observations per cell). The 18-noise mean curves of MP-SENet and Demucs are not shipped in-tree, so their rows use the per-layer fit tables, which carry α, β and the empirical clean-end value c_high; that is enough for r, ρ, f, r_c and the identity, while PC1 is read from the shipped `summary_jobB.csv`.

In [ ]:
from scipy import stats

from se_probe.layers import probed_layers
from se_probe.profiles import identity_r, saturation_spread, tradeoff_summary

summary = pd.read_csv(TABLES / "snr/summary_jobB.csv").set_index("model")

grid = pd.read_csv(TABLES / "snr/cka_heatmap_values_raw_jobB18.csv")
grid = grid[grid["layer"].str.endswith(".norm1")]
curves_muse = grid.pivot(index="layer", columns="snr", values="mean")
curves_muse = curves_muse.loc[probed_layers("muse", curves_muse.index)]
assert (grid["count"] == 824 * 18).all()

def row_from_fits(model):
    f = pd.read_csv(TABLES / f"snr/fits_{model}_snr_jobB.csv")
    a, b, ch = f["alpha"].to_numpy(), f["beta"].to_numpy(), f["c_high"].to_numpy()
    r_c = float(np.corrcoef(a, ch)[0, 1])
    fs = saturation_spread(a, ch)
    return dict(n_layers=len(f), r_alpha_beta=float(np.corrcoef(a, b)[0, 1]),
                rho_alpha_beta=float(stats.spearmanr(a, b).statistic), sat_freedom=fs,
                r_alpha_chigh=r_c, pc1=float(summary.loc[model, "pc1"]),
                floor_abs_r=float(np.sqrt(1 - fs ** 2)), predicted_r=identity_r(r_c, fs))

rows = {("MUSE", "SNR"): tradeoff_summary(curves_muse),
        ("MP-SENet", "SNR"): row_from_fits("mpsenet"),
        ("Demucs", "SNR"): row_from_fits("demucs")}
print(f"MUSE SNR r(alpha,beta) = {rows[('MUSE','SNR')]['r_alpha_beta']:+.3f}   (paper Table I: -0.950)")

## Reverberation-axis rows

Table I's C50 rows are the dereverberation fine-tunes of each architecture on the 88-RIR six-room AIR pool at 824 utterances. `mean_curves_c50_six_arms.csv` holds the per-layer mean level curves of all six arms (pre/ft × three models); when it is absent the cell falls back to the α, β and c_high of the fine-tuned arms in `c50_824_per_layer_supp.csv`, which gives every column except PC1.

In [ ]:
six = TABLES / "c50/mean_curves_c50_six_arms.csv"
c50_pc1_available = six.exists()
if c50_pc1_available:
    mc = pd.read_csv(six)
    level_cols = [c for c in mc.columns if c not in ("scale", "arm", "model", "layer", "depth")]
    for model, name in [("muse", "MUSE"), ("mpsenet", "MP-SENet"), ("demucs", "Demucs")]:
        t = mc[mc["arm"] == f"{model}_ft"].sort_values("depth")
        assert list(t["layer"]) == probed_layers(model, t["layer"])
        curves = t.set_index("layer")[level_cols]
        curves.columns = curves.columns.astype(float)
        rows[(name, "C50")] = tradeoff_summary(curves)
    print("C50 rows recomputed from the six-arm mean curves")
else:
    supp = pd.read_csv(TABLES / "c50/c50_824_per_layer_supp.csv")
    for model, name in [("muse", "MUSE"), ("mpsenet", "MP-SENet"), ("demucs", "Demucs")]:
        t = supp[supp["arm"] == f"{model}_ft"].sort_values("depth")
        a, b, ch = t["alpha"].to_numpy(), t["beta"].to_numpy(), t["c_high"].to_numpy()
        r_c = float(np.corrcoef(a, ch)[0, 1])
        fs = saturation_spread(a, ch)
        rows[(name, "C50")] = dict(n_layers=len(t), r_alpha_beta=float(np.corrcoef(a, b)[0, 1]),
                                   rho_alpha_beta=float(stats.spearmanr(a, b).statistic), sat_freedom=fs,
                                   r_alpha_chigh=r_c, pc1=np.nan, floor_abs_r=float(np.sqrt(1 - fs ** 2)),
                                   predicted_r=identity_r(r_c, fs))
    print("mean_curves_c50_six_arms.csv not found: C50 rows from the FT arms of c50_824_per_layer_supp.csv (PC1 unavailable)")

In [ ]:
table1 = pd.DataFrame([dict(Model=m, Axis=ax, n_l=r["n_layers"], r=r["r_alpha_beta"], rho=r["rho_alpha_beta"],
                            f=r["sat_freedom"], PC1=r["pc1"], r_c=r["r_alpha_chigh"],
                            floor=r["floor_abs_r"], identity_r=r["predicted_r"])
                       for (m, ax), r in rows.items()])
order = [("MUSE", "SNR"), ("MUSE", "C50"), ("MP-SENet", "SNR"), ("MP-SENet", "C50"), ("Demucs", "SNR"), ("Demucs", "C50")]
table1 = table1.set_index(["Model", "Axis"]).loc[order].reset_index()

PAPER = {("MUSE", "SNR"): (-0.950, -0.891, 0.275, 97.54, +0.823),
         ("MUSE", "C50"): (-0.993, -0.996, 0.037, 96.52, +0.860),
         ("MP-SENet", "SNR"): (-0.988, -0.929, 0.137, 98.48, +0.853),
         ("MP-SENet", "C50"): (-0.999, -0.905, 0.025, 99.23, +0.918),
         ("Demucs", "SNR"): (-0.877, -0.782, 0.956, 76.32, -0.327),
         ("Demucs", "C50"): (-0.918, -0.927, 0.167, 83.58, +0.584)}
for _, t in table1.iterrows():
    r, rho, f, pc1, r_c = PAPER[(t.Model, t.Axis)]
    print(f"{t.Model:9s} {t.Axis:4s} r {t.r:+.3f} (paper {r:+.3f}) | rho {t.rho:+.3f} ({rho:+.3f}) | "
          f"f {t.f:.3f} ({f:.3f}) | PC1 {t.PC1:6.2f} ({pc1:.2f}) | r_c {t.r_c:+.3f} ({r_c:+.3f})")
    assert abs(t.r - r) < 5e-4 and abs(t.rho - rho) < 5e-4 and abs(t.f - f) < 5e-4 and abs(t.r_c - r_c) < 5e-4
    if not np.isnan(t.PC1):
        assert abs(t.PC1 - pc1) < 0.01  # Table I prints two decimals
table1.round(3)

Every row has r_c f < 1, so the identity forces the negative sign in all six cases; what is less constrained is the magnitude. At a given f the identity permits |r| as small as √(1 − f²); Demucs under noise is the loosest row (f = 0.956, floor 0.294) and still measures 0.877, so its two parameters are genuinely separable. The five more saturated rows sit at or slightly below their floor, because the identity is exact only for perfectly linear curves.

In [ ]:
d = table1.set_index(["Model", "Axis"]).loc[("Demucs", "SNR")]
print(f"Demucs SNR: f = {d.f:.3f}, permitted floor sqrt(1 - f^2) = {d.floor:.3f}   (paper: 0.294); measured |r| = {abs(d.r):.3f}   (paper: 0.877)")
assert abs(d.floor - 0.294) < 5e-4
table1["r_c_times_f"] = table1["r_c"] * table1["f"]
print(f"max r_c * f over the six rows = {table1['r_c_times_f'].max():.3f}   (paper: nowhere exceeds 0.227)")
table1[["Model", "Axis", "r", "floor", "identity_r", "r_c_times_f"]].round(3)

## The tradeoff scatter (Fig. 7)

One point per probed layer; the noise panel uses the pretrained checkpoints, the reverberation panel the dereverberation fine-tunes.

In [ ]:
import matplotlib.pyplot as plt

from se_probe.plotting import tradeoff_scatter

snr_tabs = {m: pd.read_csv(TABLES / f"snr/fits_{m}_snr_jobB.csv") for m in ["muse", "mpsenet", "demucs"]}
ft = pd.read_csv(TABLES / "c50/fig_v5_c50_824_ft_profiles_values.csv")
c50_tabs = {m: ft[ft["model"] == m].sort_values("depth") for m in ["muse", "mpsenet", "demucs"]}
stats_snr = {m: rows[(MODEL_LABELS[m], "SNR")] for m in snr_tabs}
stats_c50 = {m: rows[(MODEL_LABELS[m], "C50")] for m in c50_tabs}

fig, axes = plt.subplots(2, 1, figsize=(6.5, 8))
tradeoff_scatter(axes[0], snr_tabs, unit="SNR", annotate_stats=stats_snr)
axes[0].set_title("(a) Additive noise, pretrained checkpoints")
tradeoff_scatter(axes[1], c50_tabs, unit="C50", annotate_stats=stats_c50)
axes[1].set_title("(b) Reverberation, dereverberation fine-tunes")
for ax in axes:
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.3 * (hi - lo))
plt.tight_layout();

## What the identity permits

`identity_r(r_c, f)` evaluates the forced correlation. As f → 0 (all layers saturate to the same clean-end value) r(α, β) → −1 whatever the data contain; as f grows the two parameters decouple and the permitted range widens.

In [ ]:
fs = np.linspace(0.01, 1.2, 200)
fig, ax = plt.subplots(figsize=(6, 3.2))
for r_c in (-0.5, 0.0, 0.5, 0.9, 1.0):
    ax.plot(fs, [identity_r(r_c, f) for f in fs], label=fr"$r_c$ = {r_c:+.1f}")
ax.plot(fs, -np.sqrt(np.clip(1 - fs ** 2, 0, None)), "k--", lw=1, label=r"floor $-\sqrt{1-f^2}$")
for _, t in table1.iterrows():
    ax.plot(t.f, t.r, marker="o", ms=5, color=MODEL_COLORS[{"MUSE": "muse", "MP-SENet": "mpsenet", "Demucs": "demucs"}[t.Model]],
            mec="k" if t.Axis == "C50" else "none")
ax.set_xlabel("saturation spread f = sd(c_high) / sd(alpha)")
ax.set_ylabel(r"$r(\alpha, \beta)$")
ax.set_title("The identity, with the six measured rows (black edge = C50)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout();